In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc ffsim matplotlib numpy pyscf qiskit-addon-sqd
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Sample-basierti Quantediagonalisierig vo emene Chemie-Hamiltonian

*Schätzige zum Recheufwand: under eim Minüüt uf emem Heron-r2-Prozessor (ACHTUNG: Das isch nur e Schätzig. Dini Laufziit cha abwiche.)*
## Hintergrundinformatione
I däm Tutorial zeige mir, wie mer rüschbehafti Quantesamples nachbearbeite cha, zum e Näheriig vom Grundzuestand vom Stickstoffmolekül $\text{N}_2$ bim Gliichgewichtsbindungsabstand z'finde. Mir verwände derfür de [sample-basierte Quantediagonalisierigalgorithmus (SQD)](https://arxiv.org/abs/2405.05068) mit Samples us emem 59-Qubit-Quanteschaltkreis (52 System-Qubits + 7 Ancilla-Qubits). E Qiskit-basierti Implementierig isch i de [SQD Qiskit Addons](https://github.com/Qiskit/qiskit-addon-sqd) verfüegbar. Meh Details finsch i de entsprechende [Docs](/guides/qiskit-addons-sqd) sowie emem [eifache Bispiil](/guides/qiskit-addons-sqd-get-started) zum Iisteige.

SQD isch e Technik zum Finde vo Eigewärt und Eigevektore vo Quanteoperatore, wie zum Bispiil em Hamiltonian vomene Quantesystem. Derfür setzt si Quante- und verteilets klassisches Rechene zämme ii. Klassischs verteiltes Rechene wird bruucht, zum Samples vo emem Quanteprozessor z'verarbeite und emene Ziil-Hamiltonian i emem vo ihne ufgspannte Unterruum z'projiziere und diagonalisiere. Das macht SQD robust gegenüber Samples, wo vo Quanterüsche korrumpiert worde sind, und ermöglicht s'Umgah mit grosse Hamiltonians — zum Bispiil Chemie-Hamiltonians mit Millione vo Wächselwürkigstärme — witer us em Erreichbare vo jedere exakte Diagonalisierigsmethode. E SQD-basierter Workflow het folgendi Schritt:

1. Wähl e Schaltkreis-Ansatz und wänd ihn uf emem Quantecomputer uf emem Referenzzuestand aa (i däm Fall em [Hartree-Fock](https://en.wikipedia.org/wiki/Hartree%E2%80%93Fock_method)-Zuestand).
2. Sample Bitfolge us em resultierete Quantezuestand.
3. Füehr d'*selbstkonsistenti Konfigurationswiederherstellig* uf de Bitfolge dür, zum d'Näheriig vomene Grundzuestand z'erhalte.

SQD funktioniert bekanntermaasse guet, wenn de Ziil-Eigezuestand dünn bsetzt isch: D'Wellefunktion isch uf eme Satz vo Basiszueständ $\mathcal{S} = {|x\rangle }$ gstützt, wo sini Grösse nöd exponentiell mit der Grösse vom Problem zunimmt.

### Quantechemie
D'Eigenschafte vo Molekül wärde grossteils dur d'Struktur vo de Elektronen i ihne bstimmt. Als fermionischi Teilche chöi Elektronen mit emem mathematische Formalismus bschriibe wärde, wos als zwöiti Quantisierig bezeichnet wird. D'Idee isch, dass es e Azahl vo *Orbitale* git, wo jeweils entweder leer oder vo emem Fermion bsetzt sind. E System vo $N$ Orbitalen wird dur e Satz vo fermionische Vernichtungsoperatore ${\hat{a}_p}_{p=1}^N$ bschriibe, wo d'fermionische Antivertauschigrelatiône erfülle:

$$
\begin{align*}
\hat{a}_p \hat{a}_q + \hat{a}_q \hat{a}_p &= 0, \\
\hat{a}_p \hat{a}_q^\dagger + \hat{a}_q^\dagger \hat{a}_p &= \delta_{pq}.
\end{align*}
$$

Dr Adjungierte $\hat{a}_p^\dagger$ wird Erzeugungsoperator gnennt.

Bis jetzt het üseri Darstellig de Spin nöd berücksichtigt — e fundamentale Eigenschaft vo Fermione. Wenn mer de Spin mit iinimmt, chöme d'Orbitale in Paar vor, wo *Raumorbitale* gnennt wärde. Jedes Raumorbital besteit us zwöi *Spinorbitalen* — eis mit de Bezeichnig «Spin-$\alpha$» und eis mit «Spin-$\beta$». Mir schriibe $\hat{a}_{p\sigma}$ für de Vernichtungsoperator, wo zum Spinorbital mit Spin $\sigma$ ($\sigma \in {\alpha, \beta}$) im Raumorbital $p$ ghört. Wenn $N$ d'Azahl vo Raumorbitale isch, git es isgsamnt $2N$ Spinorbitale. Dr Hilbert-Ruum vo däm System wird vo $2^{2N}$ orthonormierte Basisvektore ufgspannt, wo mit zweiteilige Bitfolge $\lvert z \rangle = \lvert z_\beta z_\alpha \rangle = \lvert z_{\beta, N} \cdots z_{\beta, 1} z_{\alpha, N} \cdots z_{\alpha, 1} \rangle$ bezeichnet wärde.

Dr Hamiltonian vomene Molekülsystem cha gschriibe wärde als

$$
\hat{H} = \sum_{ \substack{pr\\\sigma} } h_{pr} \, \hat{a}^\dagger_{p\sigma} \hat{a}_{r\sigma}
+ \frac12
\sum_{ \substack{prqs\\\sigma\tau} }
h_{prqs} \,
\hat{a}^\dagger_{p\sigma}
\hat{a}^\dagger_{q\tau}
\hat{a}_{s\tau}
\hat{a}_{r\sigma},
$$

wo $h_{pr}$ und $h_{prqs}$ komplexe Zahle sind, wo Molekülintegrale gnennt wärde und vo de Spezifikatione vom Molekül mit emem Computerprogramm berechnet wärde chöi. I däm Tutorial berechne mir d'Integrale mit em [PySCF](https://pyscf.org/)-Softwarepaket.

Für Details derzue, wie dr molekulare Hamiltonian abgeleitet wird, lueg in es Läehrbüech über Quantechemie ine (zum Bispiil *Modern Quantum Chemistry* vo Szabo und Ostlund). Für e übergeordneti Erchlärig, wie Quantechemieprobleem uf Quantecomputer abgbildet wärde, lueg di d'Vorlesig [*Mapping Problems to Qubits*](https://youtube.com/watch?v=TyFU6r8uEsE&t=900) vo der Qiskit Global Summer School 2024 aa.

### Lokaler unitärer Cluster-Jastrow (LUCJ)-Ansatz
SQD bruucht e Quanteschaltkreis-Ansatz, um Samples drus z'ziehe. I däm Tutorial verwände mir de [lokale unitäre Cluster-Jastrow (LUCJ)](https://pubs.rsc.org/en/content/articlelanding/2023/sc/d3sc02516k)-Ansatz, wel er physikalisch motiviert und gliichzitig hardwarefrindlich isch.

Dr LUCJ-Ansatz isch e spezialisiertè Form vomene allgemeinen unitären Cluster-Jastrow (UCJ)-Ansatz, wo d'Form het:

$$
  \lvert \Psi \rangle = \prod_{\mu=1}^L e^{\hat{K}_\mu} e^{i \hat{J}_\mu} e^{-\hat{K}_\mu} | \Phi_0 \rangle
$$

wo $\lvert \Phi_0 \rangle$ e Referenzzuestand isch (oft dr Hartree-Fock-Zuestand), und d'$\hat{K}_\mu$ und $\hat{J}_\mu$ d'Form hend:

$$
\hat{K}_\mu = \sum_{pq, \sigma} K_{pq}^\mu \, \hat{a}^\dagger_{p \sigma} \hat{a}^{\phantom{\dagger}}_{q \sigma}
\;,\;
\hat{J}_\mu = \sum_{pq, \sigma\tau} J_{pq,\sigma\tau}^\mu \, \hat{n}_{p \sigma} \hat{n}_{q \tau}
\;,
$$

wo mir de Zahloperator definiert hend als

$$
\hat{n}_{p \sigma} = \hat{a}^\dagger_{p \sigma} \hat{a}^{\phantom{\dagger}}_{p \sigma}.
$$

Dr Operator $e^{\hat{K}_\mu}$ isch e Orbitalrotiig, wo sich mit bekannte Algorithme in linearer Tiifen und linearer Verbindigkeit implementiere laat.
Zum Implementiere vom $e^{i \mathcal{J}_k}$-Tärm vomene UCJ-Ansatz bruucht mer entweder All-to-all-Verbindigkeite oder e fermionisches Swap-Netz, was für rüschbehafti, nöd fählertoleranti Quanteprozessore mit beschränkter Verbindigkeite schwierig isch. D'Idee vomene *lokale* UCJ-Ansatz isch, Dünnbsetzigsbeschränkige an d'$\mathbf{J}^{\alpha\alpha}$- und $\mathbf{J}^{\alpha\beta}$-Matrize z'lege, wo s'Implementiere in konstanter Tiifen uf Qubit-Topologiee mit beschränkter Verbindigkeite ermögliche. D'Beschränkige wärde dur e Lischt vo Indices agee, wo aaluege, welchi Matrizenträge im obere Dreick ungleich null si dörfe (da d'Matrize symmetrisch sind, muss nur s'obere Dreick aagee wärde). Die Indices chöi als Paare vo Orbitale interpretiiert wärde, wo mitänand wächselwürke dörfe.

Als Bispiil luege mir e quadratischi Gitter-Qubit-Topologie aa. Mir chöi d'$\alpha$- und $\beta$-Orbitale in parallele Linie uf em Gitter platziere, mit Verbindigkeite zwüsche dene Linie, wo «Sprossen» vomene Läitermuscht bilde, ettewää so:

![Qubit-Zuordnigsdiagrämm für de LUCJ-Ansatz uf emem quadratische Gitter](../docs/images/tutorials/improving-energy-estimation-of-a-fermionic-hamiltonian-with-sqd/baad4e53-5bfd-4cb4-8027-2ec26d27ecdd.avif)

Mit däm Setup sind Orbitale mit em glyche Spin mit ere Linientopologie verbunde, während Orbitale mit unterschiedlichem Spin verbunde sind, wenn si dasselb Raumorbital teiiled. Das git d'folgendi Indexbeschränkige für d'$\mathbf{J}$-Matrize:

$$
\begin{align*}
\mathbf{J}^{\alpha\alpha} &: \set{(p, p+1) \; , \; p = 0, \ldots, N-2} \\
\mathbf{J}^{\alpha\beta} &: \set{(p, p) \;, \; p = 0, \ldots, N-1}
\end{align*}
$$

Anders gseit: Wenn d'$\mathbf{J}$-Matrize nur an de aagäbene Indices im obere Dreick ungleich null sind, cha dr $e^{i \mathcal{J}_k}$-Tärm uf ere quadratische Topologie ohne Swap-Gates in konstanter Tiifen implementiert wärde. Natürlich macht das de Ansatz weniger ausdrucksstark, drum wärde möglicherwiis meh Ansatz-Widerholige bruucht.

D'IBM&reg;-Hardware het e Heavy-Hex-Gitter-Qubit-Topologie. Dert chame es «Zickzack»-Muscht verwände, wie unten dargstellt. I däm Muscht wärde Orbitale mit em glyche Spin uf Qubits mit ere Linientopologie abgbildet (roti und blaui Chraise), und e Verbindig zwüsche Orbitalen mit unterschiedlichem Spin isch bi jedem 4. Raumorbital vorhanden — ermöglicht dur e Ancilla-Qubit (violetti Chraise). In däm Fall sind d'Indexbeschränkige:

$$
\begin{align*}
\mathbf{J}^{\alpha\alpha} &: \set{(p, p+1) \; , \; p = 0, \ldots, N-2} \\
\mathbf{J}^{\alpha\beta} &: \set{(p, p) \;, \; p = 0, 4, 8, \ldots (p \leq N-1)}
\end{align*}
$$

![Qubit-Zuordnigsdiagrämm für de LUCJ-Ansatz uf emem Heavy-Hex-Gitter](../docs/images/tutorials/improving-energy-estimation-of-a-fermionic-hamiltonian-with-sqd/7e0ee7e1-2d24-417f-ac59-25c58db79aa9.avif)

### Selbstkonsistenti Konfigurationswiederherstellig
D'selbstkonsistenti Konfigurationswiederherstellig isch drum entwicklet worde, damit mer so vill Signal wie möglich us rüschbehaftne Quantesamples herauszieht. Well dr molekulare Hamiltonian d'Teilchezahl und den Spin Z konserviert, macht's Sinn, e Schaltkreis-Ansatz z'wähle, wo au die Symmetriee konserviert. Wenn mer ihn uf em Hartree-Fock-Zuestand aawändet, het dr resultierete Zuestand im rüschfreiee Fall e fixierti Teilchezahl und nen fixierter Spin Z. Drum sötted d'Spin-$\alpha$- und Spin-$\beta$-Hälfte vo jedere Bitfolge, wo us däm Zuestand gsampled wird, dasselb [Hamming-Gwicht](https://en.wikipedia.org/wiki/Hamming_weight) haa wie im Hartree-Fock-Zuestand. Dur d'Prääsenz vo Rüschen in aktuelle Quanteprozessore wärde manche gmässeni Bitfolge disi Eigenschaft verlätze. E eifachi Form vo Nachauswahl würd die Bitfolge verwerfe, aber das isch verschwänderisch, denn si könnted trotzdem no Signal enthalte. D'selbstkonsistenti Wiederherstelligsvorgehenswiis versuecht, nes Teil vo däm Signal i dr Nachbearbeitig z'rette. D'Vorgehenswiis isch iterativ und bruucht als Iigab e Schätzig vo de mittlere Besetzigszahle vo jedem Orbital im Grundzuestand, wo zerscht us de rohe Samples berechnet wird. D'Vorgehenswiis wird i eme Schalter uf gfüehrt, und jedi Iteration het folgendi Schritt:

1. Für jedi Bitfolge, wo d'aagäbene Symmetriee verletzt, flip iri Bits mit emem probabilistischen Verfahre, wo d'Bitfolge nöcher zur aktuelle Schätzig vo de mittlere Orbitalbesetzige bringe soll, um e neui Bitfolge z'erhalte.
2. Sammel alli alte und neie Bitfolge, wo d'Symmetriee erfülle, und wähl Teilmänge vo nere fixierte Grösse us, wo im Voraus bstimmt wird.
3. Für jedi Teilmängi vo Bitfolge, projizier de Hamiltonian in de Unterruum, wo vo de entsprechende Basisvektore ufgspannte wird (lueg im [vorherige Abschnitt](#quantum-chemistry) für e Beschriibig vo dene Basisvektore), und berechne uf emem klassischen Computer e Grundzuestandsschätzig vomene projizierten Hamiltonian.
4. Aktualisier d'Schätzig vo de mittlere Orbitalbesetzige mit der Grundzuestandsschätzig, wo d'tiefschti Energie het.

### SQD-Workflow-Diagramm
Dr SQD-Workflow isch im folgenden Diagramm dargstellt:

![Workflow-Diagramm vomene SQD-Algorithmus](../docs/images/tutorials/improving-energy-estimation-of-a-fermionic-hamiltonian-with-sqd/fd7e816f-4e2e-4dd7-a7da-f71afb9ca68d.avif)
## Vorussetzige
Bevor du mit däm Tutorial aafangsch, sorg derfür, dass du Folgendes installiert hesch:

- Qiskit SDK v1.0 oder nöier, mit [Visualisierigssupport](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 oder nöier (`pip install qiskit-ibm-runtime`)
- SQD Qiskit Addon v0.11 oder nöier (`pip install qiskit-addon-sqd`)
- ffsim v0.0.58 oder nöier (`pip install ffsim`)
## Setup

In [1]:
import math

import ffsim
import matplotlib.pyplot as plt
import numpy as np
import pyscf
import pyscf.cc
import pyscf.mcscf
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.primitives import StatevectorSampler
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

## Schritt 1: Klassischi Iigabe uf es Quanteproblem abbilden
I däm Tutorial wärde mir e Näheriig zum Grundzuestand vomene Moleküls im Gliichgewicht im cc-pVDZ-Basissatz suuche. Zerscht spezifiziere mir s'Molekül und sini Eigenschafte.

In [2]:
# Specify molecule properties
spin_sq = 0

# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (1.0, 0, 0)]],
    basis="sto-6g",
    symmetry="Dooh",
)

# Define active space
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
norb = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
n_alpha = (n_electrons + mol.spin) // 2
n_beta = (n_electrons - mol.spin) // 2
nelec = (n_alpha, n_beta)
cas = pyscf.mcscf.CASCI(scf, norb, nelec)
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), norb)

# Compute exact energy using FCI
reference_energy = cas.run().e_tot

print(f"norb = {norb}")
print(f"nelec = {nelec}")

converged SCF energy = -108.464957764796
CASCI E = -108.595987350986  E(CI) = -32.4115475088426  S^2 = 0.0000000
norb = 8
nelec = (5, 5)


Before constructing the LUCJ ansatz circuit, we first perform a CCSD calculation in the following code cell. The [$t_1$ and $t_2$ amplitudes](https://en.wikipedia.org/wiki/Coupled_cluster#Cluster_operator) from this calculation will be used to initialize the parameters of the ansatz.

In [3]:
# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()
t1 = ccsd.t1
t2 = ccsd.t2

E(CCSD) = -108.5933309085008  E_corr = -0.1283731437052354


Bevor mir de LUCJ-Ansatz-Schaltkreis konstruiere, füehre mir zerscht e CCSD-Berechniig in der folgenden Code-Zelle dür. D'[$t_1$- und $t_2$-Amplituden](https://en.wikipedia.org/wiki/Coupled_cluster#Cluster_operator) us dere Berechniig wärde bruucht, zum d'Parameter vomene Ansatzes z'initialisiere.

In [4]:
import warnings

from qiskit.transpiler import CouplingMap

warnings.formatwarning = lambda msg, *args, **kwargs: f"Warning: {msg}\n"

# Set ansatz properties
n_reps = 1
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = None  # Let generate_lucj_pass_manager determine the alpha-beta interactions

# Initialize backend
coupling_map = CouplingMap.from_heavy_hex(3)
backend = GenericBackendV2(
    coupling_map.size(),
    coupling_map=coupling_map,
    basis_gates=["cp", "xx_plus_yy", "p", "x", "swap"],
)

# Create pass manager
pass_manager, pairs_ab = ffsim.qiskit.generate_lucj_pass_manager(
    backend=backend,
    norb=norb,
    connectivity="heavy-hex",
    interaction_pairs=(pairs_aa, pairs_ab),
    optimization_level=3,
)

# Create the LUCJ ansatz operator
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization
    optimize=True,
    # Limit the number of optimization iterations to prevent the code cell from running
    # too long. Removing this line may improve results.
    options=dict(maxiter=1000),
)

# create an empty quantum circuit
qubits = QuantumRegister(2 * norb, name="q")
circuit = QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it to the quantum circuit
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)

# apply the UCJ operator to the reference state
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
circuit.measure_all()

### Step 2: Optimize for quantum hardware execution

Next, we optimize the circuit for a target hardware. Typically, this step involves initializing the hardware backend and a pass manager for that backend. However, since the LUCJ ansatz is adapted to the hardware connectivity, we already performed these actions in the previous step. All that is left to do is run the pass manager on the circuit to transpile it to an ISA circuit that can be directly executed on the QPU.

In [5]:
isa_circuit = pass_manager.run(circuit)
print(f"Gate counts: {isa_circuit.count_ops()}")

Gate counts: OrderedDict({'xx_plus_yy': 86, 'p': 16, 'measure': 16, 'cp': 15, 'x': 10, 'swap': 2, 'barrier': 1})


Jetzt verwände mir [ffsim](https://github.com/qiskit-community/ffsim), zum de Ansatz-Schaltkreis z'erstelle. Da üses Molekül e closed-shell-Hartree-Fock-Zuestand het, verwände mir d'spin-balancierti Variante vomene UCJ-Ansatzes, [UCJOpSpinBalanced](https://qiskit-community.github.io/ffsim/api/ffsim.html#ffsim.UCJOpSpinBalanced). Mir übergäbe Wächselwürkigspaare, wo für e Heavy-Hex-Gitter-Qubit-Topologie passend sind (lueg im [Hintergrundsabschnitt über de LUCJ-Ansatz](#local-unitary-cluster-jastrow-lucj-ansatz)). Mir setzend `optimize=True` i dr `from_t_amplitudes`-Methode, zum d'«komprimierti» Doppelfaktorisierig vo de $t_2$-Amplituden z'aktiviere (lueg [The local unitary cluster Jastrow (LUCJ) ansatz](https://qiskit-community.github.io/ffsim/explanations/lucj.html#Parameter-initialization-from-CCSD) in der ffsim-Dokumentation für Details).

In [6]:
rng = np.random.default_rng()
sampler = StatevectorSampler(seed=rng)
job = sampler.run([isa_circuit], shots=100_000)

In [7]:
primitive_result = job.result()
pub_result = primitive_result[0]

### Step 4: Post-process and return result in desired classical format

A useful metric to judge the quality of the QPU output is the number of valid configurations returned. A valid configuration has the correct particle number and spin Z, which means that the right half of the bitstring has Hamming weight equal to the number of spin-up electrons, and the left half has Hamming weight equal to the number of spin-down electrons. The following cell computes the fraction of sampled configurations that are valid.

In [8]:
def is_valid_bitstring(
    bitstring: str, norb: int, nelec: tuple[int, int]
) -> bool:
    n_alpha, n_beta = nelec
    return (
        len(bitstring) == 2 * norb
        and bitstring[norb:].count("1") == n_alpha
        and bitstring[:norb].count("1") == n_beta
    )


bit_array = pub_result.data.meas
num_valid = sum(
    is_valid_bitstring(b, norb, nelec) for b in bit_array.get_bitstrings()
)
valid_fraction = num_valid / bit_array.num_shots
print(f"Fraction of sampled configurations that are valid: {valid_fraction}")

Fraction of sampled configurations that are valid: 1.0


Mir empfähle folgendi Schritt zum Optimiere vomene Ansatzes und zum Hardware-kompatibel Mache:

- Wähl physikalisch Qubits (`initial_layout`) vo der Ziil-Hardware us, wo dem vorher bschriibene «Zickzack»-Muscht entspreche. S'Aalege vo Qubits in däm Muscht führt zu emem effiziente hardwarekompatible Schaltkreis mit weniger Gates. Hie isch Kod, wo d'Uuswahl vomene «Zickzack»-Muschts automatisiert und e Scoring-Heuristik brucht, um d'Fähler bi em gwählte Layout z'minimiere.
- Generier e gestuften Passmanger mit der [generate_preset_pass_manager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager)-Funktion vo Qiskit mit dim `backend` und dimem `initial_layout`.
- Setz d'`pre_init`-Schtuf vo dimem gestuften Passmanager uf `ffsim.qiskit.PRE_INIT`. `ffsim.qiskit.PRE_INIT` enthält Qiskit-Transpiler-Päss, wo Gates in Orbitalrotiigene zerläge und dann d'Orbitalrotiigene zämmefüehre, was zu weniger Gates im Endschaltkreis führt.
- Füehr de Passmanager uf dimem Schaltkreis us.
<details>
<summary>Kod für d'automatisierti Uuswahl vomene «Zickzack»-Layouts</summary>

</details>

In [9]:
expected_fraction_random = (
    math.comb(norb, n_alpha) * math.comb(norb, n_beta) / 2 ** (2 * norb)
)
print(
    f"Expected fraction of valid configurations from uniformly random bitstrings: {expected_fraction_random}"
)

Expected fraction of valid configurations from uniformly random bitstrings: 0.0478515625


Now, we estimate the ground state energy of the Hamiltonian using the `diagonalize_fermionic_hamiltonian` function. This function performs the self-consistent configuration recovery procedure to iteratively refine the noisy quantum samples to improve the energy estimate. We pass a callback function so that we can save the intermediate results for later analysis. See the [API documentation](/docs/api/qiskit-addon-sqd/fermion#diagonalize_fermionic_hamiltonian) for explanations of the arguments to `diagonalize_fermionic_hamiltonian`.

Here, we use the `initial_occupancies` argument to `diagonalize_fermionic_hamiltonian` to specify the Hartree-Fock configuration as the initial guess for the orbital occupancies in the ground state. This approach is sensible for systems where the ground state has significant support on the Hartree-Fock configuration, but it might not be appropriate in other situations, though more advanced computational methods might yield better initial guesses in those cases. Specifying `initial_occupancies` also allows configuration recovery to run even if no valid configurations were sampled, as may be the case when sampling a large circuit on a noisy QPU. Without this argument, the configuration recovery would fail and raise an error if no valid configurations were provided.

In [10]:
from functools import partial

from qiskit_addon_sqd.fermion import (
    SCIResult,
    diagonalize_fermionic_hamiltonian,
    solve_sci_batch,
)

# SQD options
energy_tol = 1e-3
occupancies_tol = 1e-3
max_iterations = 5

# Eigenstate solver options
num_batches = 3
samples_per_batch = 300
symmetrize_spin = True
carryover_threshold = 1e-4
max_cycle = 200

# Use the Hartree-Fock configuration as an initial guess for the orbital occupancies
initial_occupancies = (
    np.array([1] * n_alpha + [0] * (norb - n_alpha)),
    np.array([1] * n_beta + [0] * (norb - n_beta)),
)

# Pass options to the built-in eigensolver. If you just want to use the defaults,
# you can omit this step, in which case you would not specify the sci_solver argument
# in the call to diagonalize_fermionic_hamiltonian below.
sci_solver = partial(solve_sci_batch, spin_sq=0.0, max_cycle=max_cycle)

# List to capture intermediate results
result_history = []


def callback(results: list[SCIResult]):
    result_history.append(results)
    iteration = len(result_history)
    print(f"Iteration {iteration}")
    for i, result in enumerate(results):
        print(f"\tSubsample {i}")
        print(f"\t\tEnergy: {result.energy + nuclear_repulsion_energy}")
        print(
            f"\t\tSubspace dimension: {np.prod(result.sci_state.amplitudes.shape)}"
        )


result = diagonalize_fermionic_hamiltonian(
    hcore,
    eri,
    bit_array,
    samples_per_batch=samples_per_batch,
    norb=norb,
    nelec=nelec,
    num_batches=num_batches,
    energy_tol=energy_tol,
    occupancies_tol=occupancies_tol,
    max_iterations=max_iterations,
    sci_solver=sci_solver,
    symmetrize_spin=symmetrize_spin,
    initial_occupancies=initial_occupancies,
    carryover_threshold=carryover_threshold,
    callback=callback,
    seed=rng,
)

final_energy = result.energy + nuclear_repulsion_energy
energy_error = final_energy - reference_energy
print(f"Final energy: {final_energy}")
print(f"Final energy error: {energy_error}")

Iteration 1
	Subsample 0
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 1
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 2
		Energy: -108.59275573641656
		Subspace dimension: 900
Iteration 2
	Subsample 0
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 1
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 2
		Energy: -108.59275573641656
		Subspace dimension: 900
Final energy: -108.59275573641656
Final energy error: 0.0032316145694579745


#### Visualize the results

The first plot shows that in this simulation we are already within `1 mH` of the exact answer after the first iteration (chemical accuracy is typically accepted to be `1 kcal/mol` $\approx$ `1.6 mH`). This is a small system, though, and because the samples are noiseless, configuration recovery is not needed. On a larger system run on a noisy QPU, multiple configuration recovery iterations might be needed, and the final accuracy might be worse. Generally, the energy can be improved by allowing more configuration recovery iterations or by increasing the number of samples per batch.

The second plot shows the average occupancy of each spatial orbital after the final iteration. We can see that both the spin-up and spin-down electrons occupy the first five orbitals with high probability in our solutions.

In [11]:
# Data for energies plot
x1 = range(len(result_history))
min_e = [
    min(result, key=lambda res: res.energy).energy + nuclear_repulsion_energy
    for result in result_history
]
e_diff = [abs(e - reference_energy) for e in min_e]
yt1 = [1.0, 1e-1, 1e-2, 1e-3, 1e-4]

# Chemical accuracy (+/- 1 milli-Hartree)
chem_accuracy = 0.001

# Data for avg spatial orbital occupancy
y2 = np.sum(result.orbital_occupancies, axis=0)
x2 = range(len(y2))

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot energies
axs[0].plot(x1, e_diff, label="energy error", marker="o")
axs[0].set_xticks(x1)
axs[0].set_xticklabels(x1)
axs[0].set_yticks(yt1)
axs[0].set_yticklabels(yt1)
axs[0].set_yscale("log")
axs[0].set_ylim(1e-4)
axs[0].axhline(
    y=chem_accuracy,
    color="#BF5700",
    linestyle="--",
    label="chemical accuracy",
)
axs[0].set_title("Approximated Ground State Energy Error vs SQD Iterations")
axs[0].set_xlabel("Iteration Index", fontdict={"fontsize": 12})
axs[0].set_ylabel("Energy Error (Ha)", fontdict={"fontsize": 12})
axs[0].legend()

# Plot orbital occupancy
axs[1].bar(x2, y2, width=0.8)
axs[1].set_xticks(x2)
axs[1].set_xticklabels(x2)
axs[1].set_title("Avg Occupancy per Spatial Orbital")
axs[1].set_xlabel("Orbital Index", fontdict={"fontsize": 12})
axs[1].set_ylabel("Avg Occupancy", fontdict={"fontsize": 12})

plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/sample-based-quantum-diagonalization/extracted-outputs/caffd888-e89c-4aa9-8bae-4d1bb723b35e-0.avif" alt="Output of the previous code cell" />

## Schritt 3: Uusfüehrig mit Qiskit Primitives
Nach em Optimiere vomene Schaltkreis für d'Hardwareuusfüehrig sind mir bereit, ihn uf der Ziil-Hardware z'laufe laa und Samples für d'Grundzuestandsenergischätzig z'sammle. Da mir nur eine Schaltkreis hend, verwände mir Qiskit Runtime's [Job-Uusfüehrigsmoduse](/guides/execution-modes) und füehre üserne Schaltkreis us.

In [12]:
# ------------------------------ Step 1 ------------------------------
# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (1.0, 0, 0)]],
    basis="cc-pvdz",
    symmetry="Dooh",
)

# Define active space
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
norb = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
n_alpha = (n_electrons + mol.spin) // 2
n_beta = (n_electrons - mol.spin) // 2
nelec = (n_alpha, n_beta)
cas = pyscf.mcscf.CASCI(scf, norb, nelec)
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), norb)

# Store reference energy from SCI calculation performed separately
reference_energy = -109.22802921665716

print(f"norb = {norb}")
print(f"nelec = {nelec}")

# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()
t1 = ccsd.t1
t2 = ccsd.t2

# Set ansatz properties
n_reps = 1
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = None  # Let generate_lucj_pass_manager determine the alpha-beta interactions

# Initialize backend
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)
print(f"Using backend {backend.name}")

# Create pass manager
pass_manager, pairs_ab = ffsim.qiskit.generate_lucj_pass_manager(
    backend=backend,
    norb=norb,
    connectivity="heavy-hex",
    interaction_pairs=(pairs_aa, pairs_ab),
    optimization_level=3,
)

# Create the LUCJ ansatz operator
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization
    optimize=True,
    # Limit the number of optimization iterations to prevent the code cell from running
    # too long. Removing this line may improve results.
    options=dict(maxiter=1000),
)

# create an empty quantum circuit
qubits = QuantumRegister(2 * norb, name="q")
circuit = QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it to the quantum circuit
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)

# apply the UCJ operator to the reference state
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
circuit.measure_all()


# ------------------------------ Step 2 ------------------------------

isa_circuit = pass_manager.run(circuit)
print(f"Gate counts: {isa_circuit.count_ops()}")


# ------------------------------ Step 3 ------------------------------
sampler = Sampler(mode=backend)
sampler.options.environment.job_tags = ["TUT_SQD"]
job = sampler.run([isa_circuit], shots=100_000)
primitive_result = job.result()
pub_result = primitive_result[0]


# ------------------------------ Step 4 ------------------------------

bit_array = pub_result.data.meas
num_valid = sum(
    is_valid_bitstring(b, norb, nelec) for b in bit_array.get_bitstrings()
)
valid_fraction = num_valid / bit_array.num_shots
print(f"Fraction of sampled configurations that are valid: {valid_fraction}")
expected_fraction_random = (
    math.comb(norb, n_alpha) * math.comb(norb, n_beta) / 2 ** (2 * norb)
)
print(
    f"Expected fraction of valid configurations from uniformly random bitstrings: {expected_fraction_random}"
)
# SQD options
energy_tol = 1e-3
occupancies_tol = 1e-3
max_iterations = 5

# Eigenstate solver options
num_batches = 3
samples_per_batch = 300
symmetrize_spin = True
carryover_threshold = 1e-4
max_cycle = 200

# Use the Hartree-Fock configuration as an initial guess for the orbital occupancies
initial_occupancies = (
    np.array([1] * n_alpha + [0] * (norb - n_alpha)),
    np.array([1] * n_beta + [0] * (norb - n_beta)),
)

# Pass options to the built-in eigensolver. If you just want to use the defaults,
# you can omit this step, in which case you would not specify the sci_solver argument
# in the call to diagonalize_fermionic_hamiltonian below.
sci_solver = partial(solve_sci_batch, spin_sq=0.0, max_cycle=max_cycle)

# List to capture intermediate results
result_history = []


result = diagonalize_fermionic_hamiltonian(
    hcore,
    eri,
    bit_array,
    samples_per_batch=samples_per_batch,
    norb=norb,
    nelec=nelec,
    num_batches=num_batches,
    energy_tol=energy_tol,
    occupancies_tol=occupancies_tol,
    max_iterations=max_iterations,
    sci_solver=sci_solver,
    symmetrize_spin=symmetrize_spin,
    initial_occupancies=initial_occupancies,
    carryover_threshold=carryover_threshold,
    callback=callback,
    seed=rng,
)

final_energy = result.energy + nuclear_repulsion_energy
energy_error = final_energy - reference_energy
print(f"Final energy: {final_energy}")
print(f"Final energy error: {energy_error}")

# Data for energies plot
x1 = range(len(result_history))
min_e = [
    min(result, key=lambda res: res.energy).energy + nuclear_repulsion_energy
    for result in result_history
]
e_diff = [abs(e - reference_energy) for e in min_e]
yt1 = [1.0, 1e-1, 1e-2, 1e-3, 1e-4]

# Chemical accuracy (+/- 1 milli-Hartree)
chem_accuracy = 0.001

# Data for avg spatial orbital occupancy
y2 = np.sum(result.orbital_occupancies, axis=0)
x2 = range(len(y2))

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot energies
axs[0].plot(x1, e_diff, label="energy error", marker="o")
axs[0].set_xticks(x1)
axs[0].set_xticklabels(x1)
axs[0].set_yticks(yt1)
axs[0].set_yticklabels(yt1)
axs[0].set_yscale("log")
axs[0].set_ylim(1e-4)
axs[0].axhline(
    y=chem_accuracy,
    color="#BF5700",
    linestyle="--",
    label="chemical accuracy",
)
axs[0].set_title("Approximated Ground State Energy Error vs SQD Iterations")
axs[0].set_xlabel("Iteration Index", fontdict={"fontsize": 12})
axs[0].set_ylabel("Energy Error (Ha)", fontdict={"fontsize": 12})
axs[0].legend()

# Plot orbital occupancy
axs[1].bar(x2, y2, width=0.8)
axs[1].set_xticks(x2)
axs[1].set_xticklabels(x2)
axs[1].set_title("Avg Occupancy per Spatial Orbital")
axs[1].set_xlabel("Orbital Index", fontdict={"fontsize": 12})
axs[1].set_ylabel("Avg Occupancy", fontdict={"fontsize": 12})

plt.tight_layout()
plt.show()

converged SCF energy = -108.929838385609
norb = 26
nelec = (5, 5)
E(CCSD) = -109.2177884185544  E_corr = -0.2879500329450045
Using backend ibm_boston


Removing interaction (24, 24) from the end.
Removing interaction (20, 20) from the end.


Gate counts: OrderedDict({'sx': 7039, 'rz': 6990, 'cz': 1858, 'x': 61, 'measure': 52, 'barrier': 1})
Fraction of sampled configurations that are valid: 0.02124
Expected fraction of valid configurations from uniformly random bitstrings: 9.607888706852918e-07
Iteration 1
	Subsample 0
		Energy: -109.13889134249762
		Subspace dimension: 120409
	Subsample 1
		Energy: -109.11785470455858
		Subspace dimension: 110889
	Subsample 2
		Energy: -109.13234360554011
		Subspace dimension: 130321
Iteration 2
	Subsample 0
		Energy: -109.16392179579177
		Subspace dimension: 223729
	Subsample 1
		Energy: -109.16281938332986
		Subspace dimension: 223729
	Subsample 2
		Energy: -109.16955816711932
		Subspace dimension: 233289
Iteration 3
	Subsample 0
		Energy: -109.17905772999075
		Subspace dimension: 324900
	Subsample 1
		Energy: -109.17532445048462
		Subspace dimension: 357604
	Subsample 2
		Energy: -109.1733168689756
		Subspace dimension: 348100
Iteration 4
	Subsample 0
		Energy: -109.18437778820451
		Su

<Image src="../docs/images/tutorials/sample-based-quantum-diagonalization/extracted-outputs/3858949c-a55d-4ff8-a0fc-54fb53e131b5-3.avif" alt="Output of the previous code cell" />

## Next steps

<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following material:
- [Sample-based Krylov quantum diagonalization of a fermionic lattice model](/docs/tutorials/sample-based-krylov-quantum-diagonalization) - a related tutorial using time evolution circuits instead of a variational ansatz
- [Scale SQD chemistry workflows with Dice solver](https://qiskit.github.io/qiskit-addon-sqd/how_tos/integrate_dice_solver.html) - a page showing how to use the more efficient Dice software for diagonalization
- [SQD addon API documentation](https://qiskit.github.io/qiskit-addon-sqd/apidocs/qiskit_addon_sqd.fermion.html#qiskit_addon_sqd.fermion.diagonalize_fermionic_hamiltonian) - reference for the `diagonalize_fermionic_hamiltonian` function
- [*Chemistry beyond the scale of exact diagonalization on a quantum-centric supercomputer*](https://www.science.org/doi/10.1126/sciadv.adu9991) - the paper this tutorial is based on
</Admonition>